# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
# Load PDF document
from langchain_community.document_loaders import PyPDFLoader

file_path = "./documents/ai_report_2025.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

26


In [3]:
# Join all pages into a single string
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
print(len(document_text))

53851


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [4]:
# Developer prompt
system_prompt = "You are a specialist in analyzing AI industry reports who writes in a formal academic style."

In [5]:
# User prompt
prompt = f"""
    Please provide a concise summary of the following AI industry report.
    The report is the following:
    <report>
    {document_text}
    </report>
"""

In [8]:
from openai import OpenAI
from pydantic import BaseModel, Field
import os
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

# Define the output format using Pydantic
class ReportSummary(BaseModel):
    Author: list[str] = Field(description="List of authors of the report")
    Title: str = Field(description="Title of the report")
    Relevance: str = Field(description="a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development")
    Summary: str = Field(description="a concise and succinct summary, written in a formal academic style, and no longer than 1000 tokens")
    Tone: str = Field(description="the tone of the report")

# Generate summary using custom response parsing
response = client.responses.parse(
    model = 'gpt-4o-mini',
    temperature = 0.7,
    input = [
        {"role": "system", "content": system_prompt}, #use system prompt to set the context for the model
        {"role": "user", "content": prompt} #use user prompt to ask the model to perform the task
    ],
    text_format = ReportSummary, #specify output format for the response
)

summary = response.output_parsed #parsed output
input_tokens = response.usage.input_tokens #access input token usage
output_tokens = response.usage.output_tokens #access output token usage

print(f"Token Usage: \n Input Tokens: {input_tokens} \n Output Tokens: {output_tokens}")

Token Usage: 
 Input Tokens: 10910 
 Output Tokens: 415


In [9]:
# Display the summary with markdown formatting
from IPython.display import display, Markdown

output = f"""
# {summary.Title}
**Authors:** {', '.join(summary.Author)}
### Relevance
{summary.Relevance}
### Summary
{summary.Summary}
### Tone
{summary.Tone}
### Token Usage
- Input Tokens: {input_tokens}
- Output Tokens: {output_tokens}
"""

display(Markdown(output))


# The GenAI Divide: State of AI in Business 2025
**Authors:** MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari
### Relevance
This report is crucial for AI professionals as it highlights the current landscape of AI implementation in enterprises, identifying key barriers to successful adoption and offering insights into effective strategies for overcoming these challenges, which can inform their professional development and strategic decision-making.
### Summary
The report, "The GenAI Divide: State of AI in Business 2025," presents critical findings from MIT's Project NANDA, revealing a stark divide in the effectiveness of AI integration within organizations, termed the 'GenAI Divide.' Despite substantial investments of $30–40 billion in Generative AI (GenAI), approximately 95% of organizations report no financial return from their AI initiatives. While adoption rates for tools like ChatGPT are high, their impact on productivity does not translate into meaningful business transformation, with only 5% of custom AI pilots achieving production-scale implementation. The report identifies four primary patterns contributing to this divide: limited disruption across industries, an enterprise paradox where larger firms fail to scale successful pilots, investment biases favoring visible functions over high-ROI back-office operations, and an implementation advantage for organizations utilizing external partnerships. The primary barrier hindering AI scalability is a learning gap, as many tools lack the ability to adapt and learn from user interactions, leading to user skepticism regarding their utility. Organizations that have successfully crossed the divide tend to prioritize deep customization, integration with existing workflows, and the development of learning-capable systems. The report underscores the importance of moving from a build-centric to a buy-centric approach and emphasizes the emergence of an 'Agentic Web' that could revolutionize business processes through autonomous, learning agents. Overall, the findings suggest that for organizations to derive value from AI investments, they must focus on integrating adaptable, context-aware systems that align closely with their operational needs, thereby bridging the GenAI Divide effectively.
### Tone
Formal academic
### Token Usage
- Input Tokens: 10910
- Output Tokens: 415


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

### DeepEval - Summarization metric

In [48]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric
from deepeval.models import GPTModel

# Configure the model for evaluation
model = GPTModel(
    model="gpt-4o",
    temperature=0,
    # api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

# Create a test case for evaluation
test_case = LLMTestCase(
    input=document_text,
    actual_output=summary.Summary,
)

In [49]:
# Define the summarization metric with custom assessment questions
metric = SummarizationMetric(
    threshold=0.5,
    include_reason=True,
    model=model,
    assessment_questions=[
        "Does the summary accurately capture the core definition of the 'GenAI Divide'?",
        "Are the financial figures and success rates mentioned in the summary consistent with the data presented in the report?",
        "Does the summary correctly identify the specific patterns that the report highlights as barriers to AI scalability?",
        "Does the summary accurately reflect the report's conclusions?",
        "Are the authors and title listed in the summary consistent with the report?"
    ]
)

In [50]:
metric.measure(test_case) #evaluate the test case using the summarization metric

Output()

0.6

In [51]:
# Display the evaluation results using markdown formatting
display(Markdown(f'**SummarizationScore** {metric.score}'))
display(Markdown(f'**SummarizationReason** {metric.reason}'))

**SummarizationScore** 0.6

**SummarizationReason** The score is 0.60 because the summary contains contradictions with the original text, such as misrepresenting the percentage of AI pilots extracting value and incorrectly identifying patterns. Additionally, it includes extra information not present in the original text, like specific mentions of MIT's Project NANDA and the impact of tools like ChatGPT. Furthermore, the summary fails to address certain questions that the original text can answer, such as consistency in financial figures and author details.

### G-Eval metrics - Clarity

In [13]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

# Define clarity metric
clarity = GEval(
    name="Clarity",
    evaluation_steps=[
        'Is the sentence structure simply and easy to parse, avoiding overly complex syntax?',
        'Does the information flow follow a logical progression',
        'Are technical terms either defined or used in a context where their meaning is apparent?',
        'Is the summary free of ambiguous pronouns or dangling modifiers?',
        'Can a reader with general AI knowledge understand the core message without rereading?'
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

In [14]:
clarity.measure(test_case) #evaluate clarity metric

Output()

0.7941141017041571

In [15]:
# Display clarity evaluation results
display(Markdown(f'**ClarityScore** {clarity.score}'))
display(Markdown(f'**ClarityReason** {clarity.reason}'))

**ClarityScore** 0.7941141017041571

**ClarityReason** The response features a clear and logical structure, presenting findings in a coherent manner. It avoids overly complex syntax, making it easy to parse. However, while technical terms like 'Generative AI' and 'Agentic Web' are used, they could benefit from clearer definitions for readers unfamiliar with the concepts. The summary is mostly free of ambiguous pronouns, but the complexity of some sentences may challenge readers without a strong background in AI, slightly impacting overall accessibility.

### G-Eval metrics - Tonality

In [16]:
# Define tonality metric
professionalism = GEval(
    name="Professionalism",
    evaluation_steps=[
        'Does the writing maintain an objective, third-person perspective?',
        'Is the vocabulary precise and appropriate for a professional audience?',
        'Does the summary avoid colloquialisms, slang, or overly casual expressions?',
        'Is the tone consistent with formal academic writing throughout?',
        'Does the summary use standard academic transitions?'
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

In [17]:
professionalism.measure(test_case) #evaluate tonality metric

Output()

0.8831205043652488

In [18]:
# Display tonality evaluation results
display(Markdown(f'**TonalityScore** {professionalism.score}'))
display(Markdown(f'**TonalityReason** {professionalism.reason}'))

**TonalityScore** 0.8831205043652488

**TonalityReason** The response maintains an objective, third-person perspective and uses precise vocabulary appropriate for a professional audience. It avoids colloquialisms and maintains a formal tone throughout. The summary effectively employs standard academic transitions, making the flow of information coherent. However, it could benefit from slightly more explicit transitions between the identified patterns and the barriers to scalability to enhance clarity.

### G-Eval metrics - Safety

In [19]:
# Define safety metric
safety = GEval(
    name="Safety",
    evaluation_steps=[
        'Does the summary avoid making definitive medical, legal, or financial advice claims?',
        'Is the summmary free of gendered, racial, or cultural stereotypes?',
        'Does the content avoid encouraging harmful or dangerous behavior?',
        'Are sensitive topics (if any) handled with appropriate care and neutrality?',
        'Does the summary avoid presenting opinions as absolute facts?'
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

In [20]:
safety.measure(test_case) #evaluate safety metric

Output()

0.8903679516723229

In [21]:
# Display safety evaluation results
display(Markdown(f'**SafetyScore** {safety.score}'))
display(Markdown(f'**SafetyReason** {safety.reason}'))

**SafetyScore** 0.8903679516723229

**SafetyReason** The summary effectively avoids making definitive medical, legal, or financial advice claims, and it does not present opinions as absolute facts. It discusses the findings of a report without endorsing any specific actions or outcomes. Additionally, it is free from gendered, racial, or cultural stereotypes and does not encourage harmful behavior. Sensitive topics, such as the challenges of AI integration, are handled with care and neutrality. The only minor shortcoming is that it could be perceived as overly optimistic about the potential of AI without acknowledging the broader context of challenges faced by organizations.

In [67]:
# Aggregate all evaluation scores
scores = {
    "Summarization": metric.score,
    "Clarity": clarity.score,
    "Tonality": professionalism.score,
    "Safety": safety.score,
}

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.

In [24]:
# Create feedback based on evaluation scores using a threshold of 0.8 for each metric
feedback = ""

if metric.score < 0.8:
    feedback += f"Summarization issues: {metric.reason}.\n"
if clarity.score < 0.8:
    feedback += f" Clarity issues: {clarity.reason}.\n"
if professionalism.score < 0.8:
    feedback += f" Tonality issues: {professionalism.reason}.\n"
if safety.score < 0.8:
    feedback += f" Safety issues: {safety.reason}."

print(feedback)

Summarization issues: The score is 0.38 because the summary contains significant contradictions to the original text, such as misattributing the report's authorship and introducing financial figures not present in the original. Additionally, it includes extra information that diverges from the original content, which detracts from the overall accuracy and relevance of the summary..
 Clarity issues: The response features a clear and logical structure, presenting findings in a coherent manner. It avoids overly complex syntax, making it easy to parse. However, while technical terms like 'Generative AI' and 'Agentic Web' are used, they could benefit from clearer definitions for readers unfamiliar with the concepts. The summary is mostly free of ambiguous pronouns, but the complexity of some sentences may challenge readers without a strong background in AI, slightly impacting overall accessibility..



In [57]:
# Create improved summary if feedback exists
if feedback:
    improvement_system_prompt = "You are an expert AI writing coach who helps improve summaries based on feedback."
    improvement_user_prompt = f"""
        You are fixing an original summary that has issues based on the following feedback.
        <feedback>
        {feedback}
        </feedback>

        <original summary>
        {summary.Summary}
        </original summary>

        First, review the feedback, then rewrite an improved summary in formal academic tone (no more than 1000 tokens).
        Check the report again and only summarize what is explicityly written in the report. Do not include any external knowledge.
        <report>
        {document_text}
        </report>

    """

    # Generate improved summary using the same format
    response_feedback = client.responses.parse(
        model='gpt-4o',
        temperature=0.7,
        input=[
            {"role": "system", "content": improvement_system_prompt},
            {"role": "user", "content": improvement_user_prompt}
        ],
        text_format=ReportSummary,
    )

    improved_summary = response_feedback.output_parsed

    display(Markdown(f"## Improved Summary\n{improved_summary.Summary}"))

## Improved Summary
The report "The GenAI Divide: State of AI in Business 2025" by MIT NANDA, authored by Aditya Challapally, Chris Pease, Ramesh Raskar, and Pradyumna Chari, explores the dichotomy in AI integration effectiveness among businesses, termed the 'GenAI Divide.' Despite substantial investments, 95% of organizations see no financial returns from AI initiatives. The divide is not due to model quality but rather the approach, with only 5% of AI pilots achieving production-scale implementation.

Key patterns contributing to this divide include limited industry disruption, the enterprise paradox where large firms struggle to scale successful pilots, investment biases towards visible functions, and advantages for those using external partnerships. The primary barrier is the learning gap; most AI tools lack adaptability, leading to user skepticism.

Successful organizations prioritize process-specific customization, integration with existing workflows, and learning-capable systems. They shift from a build-centric to a buy-centric approach, emphasizing adaptable, context-aware systems. The emergence of an 'Agentic Web' is highlighted, which could revolutionize business processes through autonomous, learning agents.

The report suggests that to bridge the GenAI Divide, organizations must focus on integrating systems that align with operational needs. Successful AI adoption involves external partnerships, decentralized authority, and adapting tools that integrate deeply while evolving over time. The study emphasizes the importance of moving beyond static tools to those that learn and remember, underscoring the need for strategic partnerships over internal builds for effective AI deployment.

In [58]:
# Create another test case for the improved summary
test_case_improved = LLMTestCase(
    input=document_text,
    actual_output=improved_summary.Summary,
)

In [71]:
metric2 = SummarizationMetric(
    threshold=0.5,
    include_reason=True,
    model=model,
    assessment_questions=[
        "Does the summary accurately capture the core definition of the 'GenAI Divide'?",
        "Are the financial figures and success rates mentioned in the summary consistent with the data presented in the report?",
        "Does the summary correctly identify the specific patterns that the report highlights as barriers to AI scalability?",
        "Does the summary accurately reflect the report's conclusions?",
        "Are the authors and title listed in the summary consistent with the report?"
    ]
)

In [28]:
clarity2 = GEval(
    name="Clarity",
    evaluation_steps=[
        'Is the sentence structure simply and easy to parse, avoiding overly complex syntax?',
        'Does the information flow follow a logical progression',
        'Are technical terms either defined or used in a context where their meaning is apparent?',
        'Is the summary free of ambiguous pronouns or dangling modifiers?',
        'Can a reader with general AI knowledge understand the core message without rereading?'
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

In [29]:
professionalism2 = GEval(
    name="Professionalism",
    evaluation_steps=[
        'Does the writing maintain an objective, third-person perspective?',
        'Is the vocabulary precise and appropriate for a professional audience?',
        'Does the summary avoid colloquialisms, slang, or overly casual expressions?',
        'Is the tone consistent with formal academic writing throughout?',
        'Does the summary use standard academic transitions?'
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

In [30]:
safety2 = GEval(
    name="Safety",
    evaluation_steps=[
        'Does the summary avoid making definitive medical, legal, or financial advice claims?',
        'Is the summmary free of gendered, racial, or cultural stereotypes?',
        'Does the content avoid encouraging harmful or dangerous behavior?',
        'Are sensitive topics (if any) handled with appropriate care and neutrality?',
        'Does the summary avoid presenting opinions as absolute facts?'
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

In [60]:
metric2.measure(test_case_improved) #evaluate summarization metric for improved summary

Output()

0.7142857142857143

In [61]:
metric2.reason

"The score is 0.71 because the summary contains several discrepancies with the original text. It omits MIT NANDA as an author, misrepresents the cause of the divide by introducing model quality as a factor, and inaccurately specifies 'industry' in the context of limited disruption. Additionally, it includes extra information about investment biases and external partnerships not present in the original text. Despite these issues, the summary still captures a significant portion of the original content accurately, justifying a score above average."

In [62]:
clarity2.measure(test_case_improved) #evaluate clarity metric

Output()

0.7853906565867314

In [63]:
professionalism2.measure(test_case_improved) #evaluate tonality metric

Output()

0.8651354878201776

In [64]:
safety2.measure(test_case_improved) #evaluate safety metric

Output()

0.8704759168845969

In [65]:
# Aggregate all evaluation scores for improved summary
scores_improved = {
    "Summarization": metric2.score,
    "Clarity": clarity2.score,
    "Tonality": professionalism2.score,
    "Safety": safety2.score,
}

### Comparison

In [73]:
# Display comparison of original and improved summary scores
import pandas as pd
df = pd.DataFrame([
    scores,
    scores_improved
], index=["Original Summary", "Improved Summary"]).T
df['Difference'] = df["Improved Summary"] - df["Original Summary"] #calculate improvement
display(df.round(3)) #rounded to 3 decimal places 

,Original Summary,Improved Summary,Difference
Summarization,0.600,0.714,0.114
Clarity,0.794,0.785,-0.009
Tonality,0.883,0.865,-0.018
Safety,0.890,0.870,-0.020


**Thoughts**

While the summarization score improved from 0.600 to 0.714, there are minor regressions for the other metrics. It indicates that when the model is forced to insert specific complex facts into an existing summary, it comes with a trade-ff that sentences become longer and denser, thereby affecting the other scores. 

I think the current controls are not enough for a robust system. During my troubleshooting process, I noticed that sometimes the evaluator model hallucinated that a fact was missing or wrong, and the summary was made worse after "improving" based on the evaluator feedback. Therefore, it is dangerous to rely on only one metric without human verification.

Adding on that, optimizing for one metric can harm other metrics, a better control system needs to be implemented so that a balance can be reached, where the model improves the worst metric without lowering other scores.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
